# PaySim Dataset - Data Preprocessing

This notebook prepares the PaySim dataset for machine learning.

The preprocessing steps include:

- Loading the large dataset in chunks
- Removing duplicate records
- Checking and handling missing values
- Validating numerical values
- Creating fraud-related features
- Encoding transaction types
- Selecting relevant features
- Saving the processed dataset

The original PaySim dataset is not modified.


In [27]:
import pandas as pd
import numpy as np
import os

In [28]:
input_file = "../data/paysim.csv"
output_file = "../data/paysim_processed.csv"

print("Input file exists:", os.path.exists(input_file))

Input file exists: True


In [29]:
CHUNK_SIZE = 500000

print("Chunk size:", CHUNK_SIZE)

Chunk size: 500000


In [30]:
sample = pd.read_csv(
    input_file,
    nrows=10000
)

print("Sample shape:", sample.shape)
sample.head()

Sample shape: (10000, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [31]:
sample.columns.tolist()

['step',
 'type',
 'amount',
 'nameOrig',
 'oldbalanceOrg',
 'newbalanceOrig',
 'nameDest',
 'oldbalanceDest',
 'newbalanceDest',
 'isFraud',
 'isFlaggedFraud']

In [32]:
required_columns = [
    "step",
    "type",
    "amount",
    "nameOrig",
    "oldbalanceOrg",
    "newbalanceOrig",
    "nameDest",
    "oldbalanceDest",
    "newbalanceDest",
    "isFraud",
    "isFlaggedFraud"
]

missing_columns = [
    col for col in required_columns
    if col not in sample.columns
]

print("Missing required columns:", missing_columns)

Missing required columns: []


In [33]:
if os.path.exists(output_file):
    os.remove(output_file)
    print("Old processed file removed.")
else:
    print("No previous processed file found.")

Old processed file removed.


In [34]:
feature_columns = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "isFlaggedFraud"
]

target_column = "isFraud"

In [35]:
first_chunk = True

total_input_rows = 0
total_output_rows = 0
total_duplicates = 0
total_invalid_rows = 0

In [36]:
for chunk_number, chunk in enumerate(
    pd.read_csv(input_file, chunksize=CHUNK_SIZE),
    start=1
):

    print(f"Processing chunk {chunk_number}...")
    
    total_input_rows += len(chunk)

    # ---------------------------------------
    # 1. Remove duplicate rows
    # ---------------------------------------
    
    before_duplicates = len(chunk)
    
    chunk = chunk.drop_duplicates()
    
    duplicates_removed = (
        before_duplicates - len(chunk)
    )
    
    total_duplicates += duplicates_removed

    # ---------------------------------------
    # 2. Convert numerical columns
    # ---------------------------------------
    
    numeric_columns = [
        "step",
        "amount",
        "oldbalanceOrg",
        "newbalanceOrig",
        "oldbalanceDest",
        "newbalanceDest",
        "isFraud",
        "isFlaggedFraud"
    ]

    for column in numeric_columns:
        chunk[column] = pd.to_numeric(
            chunk[column],
            errors="coerce"
        )

    # ---------------------------------------
    # 3. Remove rows with missing values
    # ---------------------------------------
    
    before_missing = len(chunk)

    chunk = chunk.dropna(
        subset=[
            "step",
            "type",
            "amount",
            "oldbalanceOrg",
            "newbalanceOrig",
            "oldbalanceDest",
            "newbalanceDest",
            "isFraud"
        ]
    )

    total_invalid_rows += (
        before_missing - len(chunk)
    )

    # ---------------------------------------
    # 4. Remove impossible negative values
    # ---------------------------------------
    
    before_invalid = len(chunk)

    chunk = chunk[
        (chunk["amount"] >= 0) &
        (chunk["oldbalanceOrg"] >= 0) &
        (chunk["newbalanceOrig"] >= 0) &
        (chunk["oldbalanceDest"] >= 0) &
        (chunk["newbalanceDest"] >= 0)
    ]

    total_invalid_rows += (
        before_invalid - len(chunk)
    )

    # ---------------------------------------
    # 5. Clean transaction type
    # ---------------------------------------
    
    chunk["type"] = (
        chunk["type"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # ---------------------------------------
    # 6. Feature Engineering
    # ---------------------------------------

    # Change in sender's balance
    chunk["origin_balance_change"] = (
        chunk["oldbalanceOrg"]
        - chunk["newbalanceOrig"]
    )

    # Change in receiver's balance
    chunk["destination_balance_change"] = (
        chunk["newbalanceDest"]
        - chunk["oldbalanceDest"]
    )

    # Difference between amount and sender balance change
    chunk["origin_error"] = (
        chunk["amount"]
        - chunk["origin_balance_change"]
    )

    # Difference between amount and receiver balance change
    chunk["destination_error"] = (
        chunk["amount"]
        - chunk["destination_balance_change"]
    )

    # Amount relative to sender balance
    chunk["amount_to_origin_balance"] = (
        chunk["amount"]
        / (chunk["oldbalanceOrg"] + 1)
    )

    # Amount relative to receiver balance
    chunk["amount_to_destination_balance"] = (
        chunk["amount"]
        / (chunk["oldbalanceDest"] + 1)
    )

    # ---------------------------------------
    # 7. Encode transaction type
    # ---------------------------------------

    chunk = pd.get_dummies(
        chunk,
        columns=["type"],
        prefix="type",
        dtype="int8"
    )

    # ---------------------------------------
    # 8. Select final columns
    # ---------------------------------------

    encoded_type_columns = [
        col for col in chunk.columns
        if col.startswith("type_")
    ]

    final_columns = (
        feature_columns
        + [
            "origin_balance_change",
            "destination_balance_change",
            "origin_error",
            "destination_error",
            "amount_to_origin_balance",
            "amount_to_destination_balance"
        ]
        + encoded_type_columns
        + [target_column]
    )

    chunk = chunk[final_columns]

    # ---------------------------------------
    # 9. Save processed chunk
    # ---------------------------------------

    chunk.to_csv(
        output_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

    total_output_rows += len(chunk)

    print(
        f"Chunk {chunk_number} completed | "
        f"Rows saved: {len(chunk)}"
    )

Processing chunk 1...
Chunk 1 completed | Rows saved: 500000
Processing chunk 2...
Chunk 2 completed | Rows saved: 500000
Processing chunk 3...
Chunk 3 completed | Rows saved: 500000
Processing chunk 4...
Chunk 4 completed | Rows saved: 500000
Processing chunk 5...
Chunk 5 completed | Rows saved: 500000
Processing chunk 6...
Chunk 6 completed | Rows saved: 500000
Processing chunk 7...
Chunk 7 completed | Rows saved: 500000
Processing chunk 8...
Chunk 8 completed | Rows saved: 500000
Processing chunk 9...
Chunk 9 completed | Rows saved: 500000
Processing chunk 10...
Chunk 10 completed | Rows saved: 500000
Processing chunk 11...
Chunk 11 completed | Rows saved: 500000
Processing chunk 12...
Chunk 12 completed | Rows saved: 500000
Processing chunk 13...
Chunk 13 completed | Rows saved: 362620


In [37]:
print("Preprocessing completed.")
print()
print("Total input rows:", total_input_rows)
print("Total output rows:", total_output_rows)
print("Duplicates removed:", total_duplicates)
print("Invalid/missing rows removed:", total_invalid_rows)

Preprocessing completed.

Total input rows: 6362620
Total output rows: 6362620
Duplicates removed: 0
Invalid/missing rows removed: 0


In [38]:
print(
    f"Processed file size: "
    f"{os.path.getsize(output_file) / (1024 * 1024):.2f} MB"
)

Processed file size: 816.37 MB


In [39]:
processed_sample = pd.read_csv(
    output_file,
    nrows=10000
)

processed_sample.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFlaggedFraud,origin_balance_change,destination_balance_change,origin_error,destination_error,amount_to_origin_balance,amount_to_destination_balance,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,isFraud
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,9839.64,0.0,-1.455192e-11,9839.64,0.057834,9839.640000,0,0,0,1,0,0
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,1864.28,0.0,1.136868e-12,1864.28,0.087731,1864.280000,0,0,0,1,0,0
2,1,181.00,181.0,0.00,0.0,0.0,0,181.00,0.0,0.000000e+00,181.00,0.994505,181.000000,0,0,0,0,1,1
3,1,181.00,181.0,0.00,21182.0,0.0,0,181.00,-21182.0,0.000000e+00,21363.00,0.994505,0.008545,0,1,0,0,0,1
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,11668.14,0.0,0.000000e+00,11668.14,0.280788,11668.140000,0,0,0,1,0,0


In [40]:
processed_sample.columns.tolist()

['step',
 'amount',
 'oldbalanceOrg',
 'newbalanceOrig',
 'oldbalanceDest',
 'newbalanceDest',
 'isFlaggedFraud',
 'origin_balance_change',
 'destination_balance_change',
 'origin_error',
 'destination_error',
 'amount_to_origin_balance',
 'amount_to_destination_balance',
 'type_CASH_IN',
 'type_CASH_OUT',
 'type_DEBIT',
 'type_PAYMENT',
 'type_TRANSFER',
 'isFraud']

In [41]:
processed_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   step                           10000 non-null  int64  
 1   amount                         10000 non-null  float64
 2   oldbalanceOrg                  10000 non-null  float64
 3   newbalanceOrig                 10000 non-null  float64
 4   oldbalanceDest                 10000 non-null  float64
 5   newbalanceDest                 10000 non-null  float64
 6   isFlaggedFraud                 10000 non-null  int64  
 7   origin_balance_change          10000 non-null  float64
 8   destination_balance_change     10000 non-null  float64
 9   origin_error                   10000 non-null  float64
 10  destination_error              10000 non-null  float64
 11  amount_to_origin_balance       10000 non-null  float64
 12  amount_to_destination_balance  10000 non-null  

In [42]:
processed_sample.isnull().sum()

step                             0
amount                           0
oldbalanceOrg                    0
newbalanceOrig                   0
oldbalanceDest                   0
newbalanceDest                   0
isFlaggedFraud                   0
origin_balance_change            0
destination_balance_change       0
origin_error                     0
destination_error                0
amount_to_origin_balance         0
amount_to_destination_balance    0
type_CASH_IN                     0
type_CASH_OUT                    0
type_DEBIT                       0
type_PAYMENT                     0
type_TRANSFER                    0
isFraud                          0
dtype: int64

In [43]:
processed_sample.duplicated().sum()

np.int64(0)

In [44]:
processed_sample["isFraud"].value_counts()

isFraud
0    9932
1      68
Name: count, dtype: int64

In [45]:
processed_sample.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFlaggedFraud,origin_balance_change,destination_balance_change,origin_error,destination_error,amount_to_origin_balance,amount_to_destination_balance,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,isFraud
count,10000.000000,1.000000e+04,1.000000e+04,1.000000e+04,1.000000e+04,1.000000e+04,10000.0,1.000000e+04,1.000000e+04,1.000000e+04,1.000000e+04,1.000000e+04,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,4.178900,1.035467e+05,8.939330e+05,9.152741e+05,9.342758e+05,1.096606e+06,0.0,-2.134106e+04,1.623306e+05,1.248878e+05,-5.878391e+04,4.388723e+04,1.136365e+04,0.194900,0.132100,0.034400,0.546500,0.092100,0.006800
std,2.479821,2.663072e+05,2.135683e+06,2.181428e+06,2.676340e+06,3.014496e+06,0.0,1.805159e+05,1.005081e+06,2.603428e+05,9.560670e+05,1.930708e+05,1.529623e+05,0.396144,0.338616,0.182263,0.497858,0.289182,0.082185
min,1.000000,2.390000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,-1.030100e+06,-1.306083e+07,-1.000000e-02,-1.788917e+07,1.724131e-05,6.342718e-05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,4.397530e+03,1.276875e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,1.067892e+03,5.561556e-02,3.772129e-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,5.000000,1.285874e+04,2.137556e+04,1.034994e+04,0.000000e+00,0.000000e+00,0.0,5.488050e+02,0.000000e+00,5.297390e+03,6.598765e+03,4.235122e-01,1.277380e+03,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
75%,7.000000,1.143825e+05,1.782719e+05,1.760934e+05,2.831067e+05,2.520552e+05,0.0,5.831355e+03,0.000000e+00,1.377862e+05,3.037612e+04,1.810848e+02,6.988965e+03,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
max,7.000000,1.000000e+07,1.293042e+07,1.301050e+07,1.951612e+07,1.916920e+07,0.0,1.000000e+07,1.826902e+07,3.776389e+06,1.319123e+07,3.776389e+06,1.000000e+07,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [46]:
np.isinf(
    processed_sample.select_dtypes(
        include=np.number
    )
).sum()

step                             0
amount                           0
oldbalanceOrg                    0
newbalanceOrig                   0
oldbalanceDest                   0
newbalanceDest                   0
isFlaggedFraud                   0
origin_balance_change            0
destination_balance_change       0
origin_error                     0
destination_error                0
amount_to_origin_balance         0
amount_to_destination_balance    0
type_CASH_IN                     0
type_CASH_OUT                    0
type_DEBIT                       0
type_PAYMENT                     0
type_TRANSFER                    0
isFraud                          0
dtype: int64

In [47]:
processed_sample[
    [
        "amount_to_origin_balance",
        "amount_to_destination_balance"
    ]
].describe()

,amount_to_origin_balance,amount_to_destination_balance
count,1.000000e+04,1.000000e+04
mean,4.388723e+04,1.136365e+04
std,1.930708e+05,1.529623e+05
min,1.724131e-05,6.342718e-05
25%,5.561556e-02,3.772129e-01
50%,4.235122e-01,1.277380e+03
75%,1.810848e+02,6.988965e+03
max,3.776389e+06,1.000000e+07


In [48]:
print(
    "Unique isFraud values:",
    sorted(
        processed_sample["isFraud"].unique()
    )
)

Unique isFraud values: [np.int64(0), np.int64(1)]


In [49]:
type_columns = [
    col
    for col in processed_sample.columns
    if col.startswith("type_")
]

print(type_columns)

['type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


In [50]:
feature_columns_final = [
    col
    for col in processed_sample.columns
    if col != "isFraud"
]

print("Number of features:", len(feature_columns_final))

print("\nFeatures:")
for feature in feature_columns_final:
    print(feature)

Number of features: 18

Features:
step
amount
oldbalanceOrg
newbalanceOrig
oldbalanceDest
newbalanceDest
isFlaggedFraud
origin_balance_change
destination_balance_change
origin_error
destination_error
amount_to_origin_balance
amount_to_destination_balance
type_CASH_IN
type_CASH_OUT
type_DEBIT
type_PAYMENT
type_TRANSFER


In [51]:
import os

feature_file = "../models/paysim_features.txt"

# Create the destination folder if it doesn't exist
os.makedirs(os.path.dirname(feature_file), exist_ok=True)

with open(feature_file, "w") as f:
    for feature in feature_columns_final:
        f.write(feature + "\n")

print("Feature list saved to:", feature_file)

Feature list saved to: ../models/paysim_features.txt


In [52]:
print("========== PREPROCESSING VALIDATION ==========")

print("Input rows:", total_input_rows)
print("Output rows:", total_output_rows)

print(
    "Rows removed:",
    total_input_rows - total_output_rows
)

print(
    "Processed file exists:",
    os.path.exists(output_file)
)

print(
    "Processed file size:",
    f"{os.path.getsize(output_file) / (1024 * 1024):.2f} MB"
)

print(
    "Number of features:",
    len(feature_columns_final)
)

print("==============================================")

========== PREPROCESSING VALIDATION ==========
Input rows: 6362620
Output rows: 6362620
Rows removed: 0
Processed file exists: True
Processed file size: 816.37 MB
Number of features: 18


## Preprocessing Summary

The PaySim dataset has been prepared for machine learning using a memory-efficient chunk-based pipeline.

The preprocessing included:

- Duplicate removal
- Missing value handling
- Numerical value validation
- Negative value validation
- Transaction type encoding
- Account balance-based feature engineering
- Transaction amount ratio features
- Target variable validation

The original dataset remains unchanged.

The processed dataset is saved as:

`../data/paysim_processed.csv`

The feature list is saved as:

`../models/paysim_features.txt`

The processed dataset will be used in the next notebook for fraud detection model development.